# Test notebook for DPD functionality

In [1]:
import warnings
import time

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import LJChain

start_time = time.perf_counter()
molecules = LJChain(
    num_mols=[108],
    lengths=[1],
    bead_sequence=["_A"],
    bead_mass={"_A": 1.0},
    bond_lengths={"_A-_A": 1.0},
    identify_connections=False,
)
mbuild_time = time.perf_counter()
print("mbuild time:", mbuild_time - start_time)

mbuild time: 0.0032745000207796693


In [3]:
molecules.identify_connections

False

## New Random Walk System Class

In [4]:
molecules.dihedral_types

set()

In [5]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 1.0 * u.Unit("nm")
ref_mass = 1.0 * u.Unit("g/mol")
ref_energy = 1.0 * u.Unit("kcal / mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.4 * u.Unit("nm**-3"),
    bond_length=1.0,
    buffer=0.5,
    base_units=ref_values_dict,
)

(108, 3)


In [6]:
system.box

Box: Lx=4.256950, Ly=4.256950, Lz=4.256950, xy=0.000000, xz=0.000000, yz=0.000000, 

## DPD Forcefield class

In [7]:
from flowermd.library.forcefields import DPD

In [8]:
A = 250000
gamma = 1500
kT = 1.0
r_cut = 1.01
bond_k = 250000
bond_r0 = 1.0
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

## Forcefield class

In [9]:
from flowermd.library import PhantomWalk

sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=dpd_ff.hoomd_forces,
    gsd_write_freq=100,
    log_write_freq=50,
    n_steps_dpd=1000,
    n_steps_fire=200,
)
end_time = time.perf_counter()
print("Total time:", end_time - start_time)

Initializing simulation state from a gsd.hoomd.Frame.
Step 75 of 1000; TPS: 801.48; ETA: 0.0 minutes
Step 150 of 1000; TPS: 1553.63; ETA: 0.0 minutes
Step 225 of 1000; TPS: 2276.89; ETA: 0.0 minutes
Step 300 of 1000; TPS: 2977.28; ETA: 0.0 minutes
Step 375 of 1000; TPS: 3666.94; ETA: 0.0 minutes
Step 450 of 1000; TPS: 4348.2; ETA: 0.0 minutes
Step 525 of 1000; TPS: 5011.02; ETA: 0.0 minutes
Step 600 of 1000; TPS: 5663.42; ETA: 0.0 minutes
Step 675 of 1000; TPS: 6315.14; ETA: 0.0 minutes
Step 750 of 1000; TPS: 6953.72; ETA: 0.0 minutes
Step 825 of 1000; TPS: 7576.45; ETA: 0.0 minutes
Step 900 of 1000; TPS: 8201.2; ETA: 0.0 minutes
Step 975 of 1000; TPS: 8823.85; ETA: 0.0 minutes
Step 50 of 200; TPS: 59031.88; ETA: 0.0 minutes
Step 125 of 200; TPS: 71592.21; ETA: 0.0 minutes
The closest particles after DPD + FIRE are:  0.9397499561309814
Total time: 0.23596433294005692


In [10]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [11]:
## option to use gmso parameterization from FF xml.
from flowermd.library.forcefields import Bead_Spring_DPD

gmso_start = time.perf_counter()
system.apply_forcefield(
    force_field=Bead_Spring_DPD(),
    r_cut=1.01,
    kT=1.0,
    speedup_by_molgraph=True,
    speedup_by_moltag=False,
)
gmso_end = time.perf_counter()
print("gmso ff time:", gmso_end - gmso_start)

gmso ff time: 0.09323504206258804


In [12]:
system.hoomd_forcefield